# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the "Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya" dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Make sure mlcroissant is available
!pip install -U mlcroissant

## 1. Data Loading
Load Croissant schema metadata and instantiate the dataset via `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Croissant schema URL (FAIR^2)
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset using the Croissant schema URL
dataset = mlc.Dataset(croissant_url)

# Print a summary of the dataset
meta = dataset.metadata
print("Name:", meta.name)
print("Description:", meta.description)

## 2. Data Overview
Explore available *record sets*, their `@id`s, and the fields/columns defined in the schema.

In Croissant, each record set and its fields/columns are uniquely identified by their `@id`. We'll list all record sets and a sample of their fields.

In [ ]:
# List all record sets in the dataset
print("Available record sets with @id:")
for rs in dataset.record_sets:
    print(f"  Record set name: {rs.name}")
    print(f"    @id: {rs.id}")
    if hasattr(rs, 'fields') and rs.fields:
        print("    Fields:")
        for field in rs.fields:
            print(f"      name: {getattr(field, 'name', '<no name>')}, @id: {field.id}")
    elif hasattr(rs, 'columns') and rs.columns:
        print("    Columns:")
        for col in rs.columns:
            print(f"      name: {getattr(col, 'name', '<no name>')}, @id: {col.id}")
    print()

## 3. Data Extraction
Select the main record set by `@id`, load its records, and convert to a pandas DataFrame for further analysis.

> **NOTE:** Below, replace `<main_recordset_id>` with the actual record set `@id` from the previous cell output. All references must use these `@id`s.

We extract all record sets to separate DataFrames, mapping each by their `@id`.

In [ ]:
# Identify record sets and extract them into DataFrames
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set @id: {record_set_id}")
    else:
        print(f"No records for record set @id: {record_set_id}")

# Let's pick the first non-empty record set for demonstration
main_recordset_id = None
for k, v in dataframes.items():
    if not v.empty:
        main_recordset_id = k
        break
if main_recordset_id is None:
    print("No non-empty record sets found!")
else:
    print("\nColumns in main record set (@id=%s):" % main_recordset_id)
    print(dataframes[main_recordset_id].columns.tolist())
    display(dataframes[main_recordset_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll select a numeric field (by `@id`) and perform typical EDA steps:
- Filter for records above a threshold
- Normalize the field
- Group by a categorical field (if available)

Make sure to use `@id` names when referencing DataFrame columns.

In [ ]:
# Choose a numeric field from this record set
if main_recordset_id is None:
    print("No data available for EDA.")
else:
    df = dataframes[main_recordset_id]

    # Try to infer a likely numeric field by checking column names; user should adjust as needed
    numeric_field_id = None
    sample_numeric_fields = ['log_likelihood', 'coefficient', 'std_error', 'p_value', 'iteration', 'value']
    for field in df.columns:
        if any(s in field.lower() for s in sample_numeric_fields):
            numeric_field_id = field
            break
    if numeric_field_id is None:
        # Default to any numeric column
        for c in df.columns:
            if pd.api.types.is_numeric_dtype(df[c]):
                numeric_field_id = c
                break

    if numeric_field_id is None:
        print("No numeric field found in this record set for EDA.")
    else:
        print(f"Using numeric field '@id': {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
        # Filter records where field value is greater than threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a group/categorical field for groupby
        group_field_id = None
        sample_group_fields = ['ward', 'county', 'gender', 'category', 'variable', 'group']
        for c in df.columns:
            if any(s in c.lower() for s in sample_group_fields):
                group_field_id = c
                break
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame('mean_' + numeric_field_id)
            print(f"\nGrouped data by {group_field_id}, mean {numeric_field_id}:")
            display(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the numeric field and, if a grouping variable is available, compare means by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_recordset_id is not None and numeric_field_id is not None and not df.empty:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated:
- How to load and overview a Croissant-described dataset with `mlcroissant`
- How to extract record sets and reference all entities by their `@id`
- Performed filtering, normalization, and group-based EDA
- Visualized numeric distributions and group statistics

👉 This workflow provides a foundation for exploring any Croissant-compatible dataset!